<a href="https://colab.research.google.com/github/sofia-seo-j/chantey_2026/blob/AIDM/BDS_Assignment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Cousre: Big Data Statistics*

S.E. Jeong (2894452)

# Assignment Part I

## Load

In [ ]:
# Packages inmported
import pandas as pd
import statsmodels.api as sm
import numpy as np

In [ ]:
# Data immported
data = pd.read_csv('/content/Assignment_BDS_25_26_data.csv')
print("\n Summary of the data:")
data.info()


## Task 1

In [ ]:
# Linear model with all the explanatory variables
X = data.drop(columns=["radius_mean"])
X = sm.add_constant(X)
y = data["radius_mean"]

model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
# t-test using the results
alpha = 0.05
pvals = model.pvalues
significant = pvals[pvals < alpha]

print("\n Significant explanatory variables (5% level) \n")
for var, p in significant.items():
    print(f"{var}: p-value = {round(p, 4)}")

## Task 2

In [ ]:
# Variable backward elimination
threshold = 3.86
while True:

    full_model = sm.OLS(y, X).fit()
    SSE_full = sum(full_model.resid**2)

    n = full_model.nobs
    d = len(full_model.params)

    F_stats = {}

    for var in X.columns:

        X_restricted = X.drop(columns=[var])
        restricted_model = sm.OLS(y, X_restricted).fit()
        SSE_restricted = sum(restricted_model.resid**2)

        F = (SSE_restricted - SSE_full) / (SSE_full / (n - d))
        F_stats[var] = F

    worst_var = min(F_stats, key=F_stats.get)
    min_F = F_stats[worst_var]

    if min_F > threshold:
        break

    print(f"Removing {worst_var} (F = {min_F:.2f})")
    X = X.drop(columns=[worst_var])

In [ ]:
model2 = sm.OLS(y, X).fit()

alpha = 0.05
pvals2 = model2.pvalues

# Keep only significant ones
significant2 = pvals2[pvals2 < alpha]

print("\nSignificant explanatory variables after backward elimination (5% level)\n")

for var, p in significant2.items():
    print(f"{var}: p-value = {round(p, 4)}")

In [ ]:
# Task 1 significant variables
task1_vars = set(significant.index)

# Task 2 significant variables
task2_vars = set(significant2.index)

print("\nVariables in Task 1 but not Task 2:")
print(task1_vars - task2_vars)

print("\nVariables in Task 2 but not Task 1:")
print(task2_vars - task1_vars)